# Optimization

In [9]:
import numpy as np
import matplotlib.pyplot as plt

In [13]:
inputs = dict(MAC = 0.4, # Mean Aerodynamic Chord [m]
AR = 14, # [-]
S = 5.5, # [m^2]
taper = 0.4, # [-]
rootchord = 0.5, # [m]
thicknessChordRatio = 0.17, # [-]
xAC = 0.25, # [-] position of ac with respect to the chord
mtom = 1972, # maximum take-off mass from statistical data - Class I estimation
S1 = 5.5,
S2 = 5.5, # surface areas of wing one and two
A = 14, # aspect ratio of a wing, not aircraft
nmax = 3.2, # maximum load factor
Pmax = 15.25, # this is defined as maximum perimeter in Roskam, so i took top down view of the fuselage perimeter
lf = 7.2, # length of fuselage
m_pax = 95, # average mass of a passenger according to Google
n_prop = 16, # number of engines
n_pax = 5, # number of passengers (pilot included)
pos_fus = 3.6, # fuselage centre of mass away from the nose
pos_lgear = 3.6, # landing gear position away from the nose
pos_frontwing = 0.2,
pos_backwing = 7, # positions of the wings away from the nose
m_prop = [30]*16, # list of mass of engines (so 30 kg per engine with nacelle and propeller)
pos_prop = [0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0], # 8 on front wing and 8 on back wing
Mac = 0.0645 * 0.5 * 1.2 * 53 ** 2 * 5.25 * 0.65, # aerodynamic moment around AC
flighttime = 3, # [hr]
turnovertime = 2, # we dont actually need this xd
takeofftime = 0.1,
enginePlacement = list(np.linspace(0.1*(14 * 5.5) ** 0.5/2, 0.8*(14 * 5.5) ** 0.5/2, 4)),
engineMass = 400 * 9.81 / 8,
Thover = 2960,
Tcruise = 2960,
p_pax = [1.5, 3, 3, 4.2, 4.2],
battery_pos = 3.6,
cargo_m = 85, cargo_pos = 6, battery_m = 400) | json.loads(open('ldists.json', 'r').read())

In [17]:
from Geometry import HatStringer, JStringer, ZStringer, WingBox, WingStructure
from SolveLoads import WingLoads, Engines, Fatigue
from Weight import *
from Material import Material


class Structure:
    def __init__(self, **inputs):
        self.__dict__.update(inputs)
        base, height = 0.75 - 0.15, 0.11571117509557907 + 0.03145738125495376 # x/c, y/c
        self.n_ult = nmax*1.5
        self.normalBox = WingBox(thicknessOfSkin, thicknessOfSpar, base, height)
        self.span = (AR * S) ** 0.5
        self.wing = WingStructure(self.span, taper, rootchord, normalBox)
        self.hatGeom = dict(bflange1 = 0.02, bflange2 =0.02, tflange = 0.02, vflange = 0.035, tstr = 0.001)
        self.omax, self.taumax, self.Ymax, self.cycles, self.matsk, self.matstr, self.loads, self.critbuckling = [None]*8

    def __setitem__(self, key, item):
        self.__dict__[key] = item

    __getitem__ = lambda self, key: self.__dict__[key]
        
    def compute_stresses(self, nStrT, nStrB, thicknessOfSkin, thicknessOfSpar):
        args = dict(span=self.span, taper=self.taper, cr=self.rootchord, tsk=thicknessOfSkin,
                    tsp=thicknessOfSpar, toc=self.thicknessChordRatio, nStrT=nStrT, nStrB=nStrB, StrA=StrA,
                    strType='Hat', strGeo=self.hatGeom, mac=MAC, xac=xAC,
                    engines=Engines(self.Thover, self.Tcruise, self.enginePlacement,self.engineMass), frac=0.6)

        self.loads = WingLoads(**args)
        wing = Wing(self.mtom, self.S1, self.S2, self.n_ult, self.A, [self.pos_frontwing, self.pos_backwing])
        fuselage = Fuselage(self.mtom, self.Pmax, self.lf, self.n_pax, self.pos_fus)
        lgear = LandingGear(self.mtom, self.pos_lgear)
        props = Propulsion(self.n_prop, self.m_prop, self.pos_prop)
        w = Weight(m_pax, wing, fuselage, lgear, props,
                   cargo_m = self.cargo_m, cargo_pos = self.cargo_pos, battery_m = self.battery_m,
                   battery_pos = self.battery_pos, p_pax = self.p_pax)
        reactionsCruise = self.loads.equilibriumCruise([self.pos, self.dragd], [self.pos, self.liftd],
                                                  [self.pos, [self.Mac / self.span]*len(self.pos)], self.wingWeight)
        reactionsVTO = self.loads.equilibriumVTO(self.wingWeight)
        lift, wgt = self.loads.internalLoads([self.pos, self.dragd], [self.pos, self.liftd],
                                        [self.pos, [self.Mac / self.span]*len(self.pos)], self.wingWeight)
        VxVTO, MyVTO = self.loads.internalLoadsVTO(self.wingWeight)
        coords, o_cr, tau_cr, Ymcr = self.loads.stressesCruise()
        coords, o_VTO, tau_VTO, YmVTO = self.loads.stressesVTO()
        self.omax, self.taumax = WingLoads.extreme(coords, o_cr)[2], WingLoads.extreme(coords, tau_cr)[2]
        self.Ymax = WingLoads.extreme(coords, Ymcr)[2]
        return self.omax, self.taumax, self.Ymax
    
    def compute_fatigue(self):
        liftdist = np.array(self.liftd) * self.n_ult
        dragdist = np.array(self.dragd) * self.n_ult
        pos, liftd, Mac, span, wingWeight = [self[k] for k in 'pos, liftd, Mac, span, wingWeight'.split(', ')]
        
        fatigue_reactionsCruise = loads.equilibriumCruise([pos, dragd], [pos, liftd], [pos, [Mac / span]*len(pos)], wingWeight)
        fatigue_lift, fatigue_wgt = loads.internalLoads([pos, dragd], [pos, liftd], [pos, [Mac / span]*len(pos)], wingWeight)
        coords, ocrf, taucrf, Ymcrf = loads.stressesCruise()

        fatigue_reactionsVTO = loads.equilibriumVTO(wingWeight)
        fatigue_VxVTO, fatigue_MyVTO = loads.internalLoadsVTO(wingWeight)
        coords, oVTOf, tauVTOf, YmVTOf = loads.stressesVTO()

        fatigue_reactionsVTOgr = loads.equilibriumVTO(wingWeight, ground = True)
        fatigue_VxVTOgr, fatigue_MyVTOgr = loads.internalLoadsVTO(wingWeight, ground = True)
        coords, oVTOfgr, tauVTOfgr, YmVTOfgr = loads.stressesVTO()

        *coor, maxDif = loads.extreme(coords, oVTOf - ocrf)
        ind = [i for i in range(len(coords)) if np.all(coords[i] == coor)][0]

        oVTOfgrmd, oVTOfmd, ocrfmd = oVTOfgr[ind], oVTOf[ind], ocrf[ind]

        fatigue = Fatigue(oVTOfgrmd, oVTOfmd, ocrfmd, flighttime, turnovertime, takeofftime, material)

        t, y = fatigue.determineCycle()
        fdf = fatigue.getCycles()
        self.cycles = fatigue.MinersRule()
        return self.cycles
    
    def compute_buckling(self, stringerMat = dict(file='materials.csv', material='Al 6061', Condition='T6'),
                         skinMat = dict(file='materials.csv', material='Al 6061', Condition='T6')):

        root = self.loads.wing(0)
        self.matstr, self.matsk = Material.load(**stringerMat), Material.load(**skinMat)
        EofStringers = self.matstr.E
        vOfStringers = self.matsk.v
        yieldOfStringers = self.matstr.oyield
        EofSkin = self.matsk.E
        vOfSkin = self.matsk.v
        self.critbuckling = root.Bstress(EofStringers, vOfStringers, yieldOfStringers, EofSkin, vOfSkin)
        return self.critbuckling
    
    def optimize(self):
        pass

        